# 05: Experiment Power Analysis

**Context:** Part 4 identified delivery reliability (`pct_orders_late`) as the strongest predictor of review score decline. Part 5 designs an experiment to test whether proactive fulfillment alerts can improve on-time delivery rates before that decline takes hold.

**Intervention:** A "Proactive Merchant Health Alerts" feature that surfaces fulfillment benchmarks and peer comparisons to merchants whose late delivery rate is trending upward.

**This notebook answers three questions:**
1. How many merchants per arm do we need to detect a 5 pp improvement in on-time delivery rate at 80% power?
2. How long would the experiment run given the eligible merchant pool?
3. Is the experiment adequately powered to catch GMV harm if the intervention backfires?

**Key design parameters (from Part 5 PRD):**
- Primary metric: 30-day change in on-time delivery rate
- Minimum detectable effect (MDE): 5 percentage points
- Power: 80% (β = 0.20)
- Significance level: α = 0.05 (two-sided)
- Randomisation unit: seller

In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.power import TTestIndPower
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

PROCESSED = Path('../data/processed')
RAW       = Path('../data/raw')

analysis = TTestIndPower()

# Experiment design constants
ALPHA      = 0.05
POWER      = 0.80
MDE_PRIMARY = 0.05   # 5 percentage points improvement in on-time delivery rate

print('statsmodels TTestIndPower ready.')
print(f'Design: alpha={ALPHA}, power={POWER}, MDE={MDE_PRIMARY*100:.0f}pp, two-sided')

statsmodels TTestIndPower ready.
Design: alpha=0.05, power=0.8, MDE=5pp, two-sided


---
## 1. Baseline: On-Time Delivery Rate

The power calculation requires the observed variance of the primary metric. We use the merchant-level standard deviation of the on-time delivery rate from merchants with 10+ orders (the same population that would be eligible for the experiment).

In [2]:
ms = pd.read_csv(PROCESSED / 'merchant_segments.csv')

# Eligible: 10+ orders (same filter as regression, gives reliable aggregated metrics)
eligible = ms[ms['order_count'] >= 10].dropna(subset=['pct_orders_late', 'total_gmv']).copy()
eligible['on_time_rate'] = 1.0 - eligible['pct_orders_late']

N_ELIGIBLE  = len(eligible)
MU_OTR      = eligible['on_time_rate'].mean()
SIGMA_OTR   = eligible['on_time_rate'].std()
MEDIAN_OTR  = eligible['on_time_rate'].median()

print(f'Eligible merchants (10+ orders): {N_ELIGIBLE:,}')
print()
print('On-time delivery rate (merchant-level):')
print(eligible['on_time_rate'].describe().round(4))
print()
print(f'Mean:   {MU_OTR:.4f}  ({MU_OTR*100:.2f}%)')
print(f'Std:    {SIGMA_OTR:.4f}  ({SIGMA_OTR*100:.2f} percentage points)')
print(f'Median: {MEDIAN_OTR:.4f}  ({MEDIAN_OTR*100:.2f}%)')
print()
print(f'Segment breakdown of eligible merchants:')
print(eligible['segment'].value_counts())

Eligible merchants (10+ orders): 1,226

On-time delivery rate (merchant-level):
count    1226.0000
mean        0.9198
std         0.0717
min         0.3571
25%         0.8889
50%         0.9312
75%         0.9697
max         1.0000
Name: on_time_rate, dtype: float64

Mean:   0.9198  (91.98%)
Std:    0.0717  (7.17 percentage points)
Median: 0.9312  (93.12%)

Segment breakdown of eligible merchants:
segment
Champions       645
Rising Stars    536
Dormant          45
Name: count, dtype: int64


In [3]:
# Visualise the baseline distribution
fig_baseline = go.Figure()

fig_baseline.add_trace(go.Histogram(
    x=eligible['on_time_rate'] * 100,
    nbinsx=40,
    marker_color='#3498db',
    opacity=0.75,
    hovertemplate='On-time rate: %{x:.1f}%<br>Count: %{y}<extra></extra>'
))

fig_baseline.add_vline(
    x=MU_OTR * 100, line_dash='solid', line_color='#e74c3c', line_width=2,
    annotation_text=f'Mean: {MU_OTR*100:.1f}%',
    annotation_position='top right'
)
fig_baseline.add_vline(
    x=(MU_OTR + MDE_PRIMARY) * 100, line_dash='dash', line_color='#2ecc71', line_width=1.5,
    annotation_text=f'Target: {(MU_OTR+MDE_PRIMARY)*100:.1f}% (+5pp)',
    annotation_position='top left'
)

fig_baseline.update_layout(
    title=dict(
        text='Baseline On-Time Delivery Rate Distribution: Merchants with 10+ Orders<br>'
             '<sup>Red line = observed mean. Green dashed = treatment target (mean + 5pp MDE).</sup>',
        font_size=15
    ),
    xaxis_title='On-Time Delivery Rate (%)',
    yaxis_title='Number of Merchants',
    plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(gridcolor='#eee'),
    yaxis=dict(gridcolor='#eee'),
    height=400, margin=dict(t=100)
)

fig_baseline.show()
print(f'Takeaway: The baseline on-time delivery rate is {MU_OTR*100:.1f}% with a standard deviation '
      f'of {SIGMA_OTR*100:.1f} percentage points. The target improvement of 5pp would move the '
      f'average merchant from {MU_OTR*100:.1f}% to {(MU_OTR+MDE_PRIMARY)*100:.1f}% on-time.')

Takeaway: The baseline on-time delivery rate is 92.0% with a standard deviation of 7.2 percentage points. The target improvement of 5pp would move the average merchant from 92.0% to 97.0% on-time.


**Takeaway:** The baseline on-time delivery rate across eligible merchants is 91.98% with a standard deviation of 7.17 percentage points. The distribution is left-skewed, meaning most merchants perform well but a meaningful tail falls below 80% on-time. The experiment would target this tail: merchants whose late delivery rate is trending upward.

---
## 2. Primary Power Analysis: 5pp MDE on On-Time Delivery Rate

In [4]:
# Cohen's d effect size = MDE / sigma
EFFECT_SIZE_PRIMARY = MDE_PRIMARY / SIGMA_OTR

N_PER_GROUP = analysis.solve_power(
    effect_size=EFFECT_SIZE_PRIMARY,
    power=POWER,
    alpha=ALPHA,
    alternative='two-sided'
)
N_PER_GROUP_CEIL = int(np.ceil(N_PER_GROUP))
N_TOTAL         = 2 * N_PER_GROUP_CEIL

print('=== Primary Power Analysis: On-Time Delivery Rate ===')
print()
print(f'Observed std (sigma):        {SIGMA_OTR:.4f}  ({SIGMA_OTR*100:.2f} pp)')
print(f'MDE:                         {MDE_PRIMARY*100:.0f} pp')
print(f"Cohen's d effect size:       {EFFECT_SIZE_PRIMARY:.4f}")
print(f'Significance level (alpha):  {ALPHA}')
print(f'Power (1 - beta):            {POWER}')
print(f'Test type:                   Two-sided independent samples t-test')
print()
print(f'Required N per group:        {N_PER_GROUP_CEIL}')
print(f'Required N total:            {N_TOTAL}')
print(f'Eligible merchant pool:      {N_ELIGIBLE:,}')
print(f'Pool utilisation:            {N_TOTAL/N_ELIGIBLE:.1%}  (only {N_TOTAL/N_ELIGIBLE:.1%} of eligible merchants needed)')

# Verify achieved power at ceiling N
achieved_power = analysis.solve_power(
    effect_size=EFFECT_SIZE_PRIMARY,
    nobs1=N_PER_GROUP_CEIL,
    alpha=ALPHA,
    alternative='two-sided'
)
print(f'\nAchieved power at N={N_PER_GROUP_CEIL}/group: {achieved_power:.4f}')

=== Primary Power Analysis: On-Time Delivery Rate ===

Observed std (sigma):        0.0717  (7.17 pp)
MDE:                         5 pp
Cohen's d effect size:       0.6970
Significance level (alpha):  0.05
Power (1 - beta):            0.8
Test type:                   Two-sided independent samples t-test

Required N per group:        34
Required N total:            68
Eligible merchant pool:      1,226
Pool utilisation:            5.5%  (only 5.5% of eligible merchants needed)

Achieved power at N=34/group: 0.8083


The 5pp MDE corresponds to a Cohen's d effect size of 0.70, which falls in the medium-to-large range by conventional standards. Because the on-time delivery rate has relatively low variance across eligible merchants (σ = 7.2pp), even a modest absolute improvement would be detectable with a fairly small sample.

---
## 3. Power Curve: Required N vs. MDE

In [5]:
MDES_PP   = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15]
MDES_FRAC = [m / 100 for m in MDES_PP]

ns_per_group = []
effect_sizes = []
achieved_powers_at_n = []

for mde in MDES_FRAC:
    es = mde / SIGMA_OTR
    n  = analysis.solve_power(effect_size=es, power=POWER, alpha=ALPHA, alternative='two-sided')
    ns_per_group.append(int(np.ceil(n)))
    effect_sizes.append(es)

power_curve_df = pd.DataFrame({
    'mde_pp':        MDES_PP,
    'mde_frac':      MDES_FRAC,
    'effect_size':   [round(e, 3) for e in effect_sizes],
    'n_per_group':   ns_per_group,
    'n_total':       [2 * n for n in ns_per_group],
})

print('Power curve (80% power, alpha=0.05):')
print(power_curve_df.to_string(index=False))

Power curve (80% power, alpha=0.05):
 mde_pp  mde_frac  effect_size  n_per_group  n_total
      1      0.01        0.139          809     1618
      2      0.02        0.279          203      406
      3      0.03        0.418           91      182
      4      0.04        0.558           52      104
      5      0.05        0.697           34       68
      6      0.06        0.836           24       48
      8      0.08        1.115           14       28
     10      0.10        1.394           10       20
     12      0.12        1.673            7       14
     15      0.15        2.091            5       10


In [6]:
# Power curve: N per group vs MDE
fig_power = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Required N per Group vs. MDE<br><sup>At 80% power, alpha=0.05, two-sided</sup>',
        'Power vs. N per Group (at 5pp MDE)<br><sup>How power grows as sample size increases</sup>'
    ),
    horizontal_spacing=0.12
)

# Left: N vs MDE
fig_power.add_trace(
    go.Scatter(
        x=power_curve_df['mde_pp'],
        y=power_curve_df['n_per_group'],
        mode='lines+markers',
        line=dict(color='#3498db', width=2),
        marker=dict(size=7),
        hovertemplate='MDE: %{x}pp<br>N per group: %{y}<extra></extra>',
        showlegend=False
    ), row=1, col=1
)

# Mark the 5pp design point
fig_power.add_trace(
    go.Scatter(
        x=[5], y=[N_PER_GROUP_CEIL],
        mode='markers',
        marker=dict(size=14, color='#e74c3c', symbol='star'),
        name=f'Design point: 5pp, N={N_PER_GROUP_CEIL}/group',
        hovertemplate=f'Design point<br>MDE=5pp, N={N_PER_GROUP_CEIL}/group<extra></extra>'
    ), row=1, col=1
)

# Right: power vs N at 5pp MDE
n_range  = np.arange(5, 200, 2)
pwr_vals = [
    analysis.solve_power(effect_size=EFFECT_SIZE_PRIMARY, nobs1=n, alpha=ALPHA, alternative='two-sided')
    for n in n_range
]

fig_power.add_trace(
    go.Scatter(
        x=n_range, y=pwr_vals,
        mode='lines',
        line=dict(color='#2ecc71', width=2),
        hovertemplate='N/group: %{x}<br>Power: %{y:.3f}<extra></extra>',
        showlegend=False
    ), row=1, col=2
)

fig_power.add_hline(y=0.80, line_dash='dash', line_color='#e74c3c', line_width=1.5,
                    annotation_text='80% power threshold',
                    annotation_position='bottom right', row=1, col=2)
fig_power.add_vline(x=N_PER_GROUP_CEIL, line_dash='dot', line_color='grey',
                    line_width=1, row=1, col=2)

fig_power.update_xaxes(title_text='Minimum Detectable Effect (percentage points)', row=1, col=1)
fig_power.update_yaxes(title_text='Required N per Group', row=1, col=1, gridcolor='#eee')
fig_power.update_xaxes(title_text='N per Group', row=1, col=2)
fig_power.update_yaxes(title_text='Statistical Power (1 - β)', row=1, col=2,
                        gridcolor='#eee', range=[0, 1.05])

fig_power.update_layout(
    title=dict(text='Power Analysis: On-Time Delivery Rate (Primary Metric)', font_size=15),
    plot_bgcolor='white', paper_bgcolor='white',
    height=420, margin=dict(t=100)
)

fig_power.show()
print(f'Takeaway: Detecting a 5pp improvement requires only N={N_PER_GROUP_CEIL} merchants per arm '
      f'(a small fraction of the {N_ELIGIBLE:,} eligible merchants). '
      f'Smaller MDEs (e.g. 3pp) require N=91/group but remain achievable within the eligible pool.')

Takeaway: Detecting a 5pp improvement requires only N=34 merchants per arm — a small fraction of the 1,226 eligible merchants. Smaller MDEs (e.g. 3pp) require N=91/group but remain achievable within the eligible pool.


**Takeaway:** The primary metric is well-powered. Detecting a 5pp improvement requires only 34 merchants per arm (68 total), and the eligible pool of 1,226 merchants is 18x larger than that. There is also room to detect smaller effects if needed (3pp requires N=91/group) without exhausting the available sample.

---
## 4. Experiment Runtime Estimate

In [7]:
# Runtime framing:
# The experiment randomises at the seller level.
# Sample size (N=68) is trivially achievable from the 1,226 eligible merchants.
# Runtime is therefore driven by the observation window, not sample accumulation.
#
# Primary metric: 30-day change in on-time delivery rate
# → Minimum observation window: 4 weeks
# → Buffer for orders to complete after window closes: +1 week
# → Analysis and decision: +1 week
#
# But we should also model a staged / conservative rollout
# where we enrol a subset of merchants per week.

OBS_WINDOW_WEEKS = 4     # 30-day primary metric window
BUFFER_WEEKS     = 2     # order completion + analysis lag

print('=== Runtime Estimate ===')
print()
print(f'Eligible merchant pool:     {N_ELIGIBLE:,}')
print(f'Required total N:           {N_TOTAL}  ({N_PER_GROUP_CEIL} per arm)')
print(f'Pool utilisation at launch: {N_TOTAL/N_ELIGIBLE:.1%}')
print()

# Scenario A: full randomisation at launch
print('Scenario A: Full randomisation at launch (enrol all eligible immediately):')
print(f'  Enrolment complete:       Day 1')
print(f'  Observation window:       {OBS_WINDOW_WEEKS} weeks')
print(f'  Buffer:                   {BUFFER_WEEKS} weeks')
print(f'  Total experiment runtime: {OBS_WINDOW_WEEKS + BUFFER_WEEKS} weeks')
print(f'  Enrolled vs required:     {N_ELIGIBLE} enrolled, {N_TOTAL} required → {N_ELIGIBLE//2}/arm '
      f'({N_ELIGIBLE//2/N_PER_GROUP_CEIL:.0f}× required N, substantial over-powering)')
print()

# Scenario B: conservative 50% enrolment rate (practical targeting window)
ENROLL_RATE = 0.50
enrolled_B  = int(N_ELIGIBLE * ENROLL_RATE)
print(f'Scenario B: 50% enrolment (targeting merchants with elevated late-delivery trend):')
print(f'  Enrolled merchants:       {enrolled_B}  ({ENROLL_RATE:.0%} of eligible pool)')
print(f'  Per arm (50/50 split):    {enrolled_B // 2}')
print(f'  Meets N={N_PER_GROUP_CEIL}/arm threshold: {enrolled_B // 2 >= N_PER_GROUP_CEIL}')
print(f'  Observation window:       {OBS_WINDOW_WEEKS} weeks')
print(f'  Buffer:                   {BUFFER_WEEKS} weeks')
print(f'  Total experiment runtime: {OBS_WINDOW_WEEKS + BUFFER_WEEKS} weeks')
print()

# Scenario C: gradual weekly rollout (most conservative)
# Approximate new merchants entering eligibility based on dataset growth rate
# Dataset spans ~24 months with 1,226 qualifying merchants → ~51/month → ~13/week
DATASET_MONTHS = 24
NEW_PER_MONTH  = N_ELIGIBLE / DATASET_MONTHS
NEW_PER_WEEK   = NEW_PER_MONTH / 4.33
print(f'Scenario C: Sequential enrolment of newly eligible merchants only:')
print(f'  Estimated new eligible merchants/week: ~{NEW_PER_WEEK:.0f}')
print(f'  Weeks to reach N={N_TOTAL} (total):         {N_TOTAL / NEW_PER_WEEK:.1f} weeks')
print(f'  Total runtime (enrolment + observation): {N_TOTAL/NEW_PER_WEEK + OBS_WINDOW_WEEKS:.1f} weeks')
print()
print('Recommended approach: Scenario B. Enrol all merchants with an elevated '
      'trailing late-rate trend at launch, observe for 4 weeks.')

=== Runtime Estimate ===

Eligible merchant pool:     1,226
Required total N:           68  (34 per arm)
Pool utilisation at launch: 5.5%

Scenario A — Full randomisation at launch (enrol all eligible immediately):
  Enrolment complete:       Day 1
  Observation window:       4 weeks
  Buffer:                   2 weeks
  Total experiment runtime: 6 weeks
  Enrolled vs required:     1226 enrolled, 68 required → 613/arm (18× required N, substantial over-powering)

Scenario B — 50% enrolment (targeting merchants with elevated late-delivery trend):
  Enrolled merchants:       613  (50% of eligible pool)
  Per arm (50/50 split):    306
  Meets N=34/arm threshold: True
  Observation window:       4 weeks
  Buffer:                   2 weeks
  Total experiment runtime: 6 weeks

Scenario C — Sequential enrolment of newly eligible merchants only:
  Estimated new eligible merchants/week: ~12
  Weeks to reach N=68 (total):         5.8 weeks
  Total runtime (enrolment + observation): 9.8 weeks

Rec

In [8]:
# Visualise runtime scenarios
scenarios   = ['A: Full pool\n(100% enrolment)', 'B: Targeted\n(50% enrolment)', 'C: Sequential\n(new merchants only)']
runtimes    = [OBS_WINDOW_WEEKS + BUFFER_WEEKS,
               OBS_WINDOW_WEEKS + BUFFER_WEEKS,
               round(N_TOTAL / NEW_PER_WEEK + OBS_WINDOW_WEEKS, 1)]
enrol_sizes = [N_ELIGIBLE, enrolled_B, N_TOTAL]
bar_colors  = ['#2ecc71', '#3498db', '#f39c12']

fig_runtime = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Experiment Runtime by Enrolment Scenario (weeks)',
        'Enrolled Merchants vs Required N'
    ),
    horizontal_spacing=0.14
)

fig_runtime.add_trace(
    go.Bar(
        x=scenarios, y=runtimes,
        marker_color=bar_colors,
        text=[f'{r} wks' for r in runtimes],
        textposition='outside',
        showlegend=False,
        hovertemplate='%{x}<br>Runtime: %{y} weeks<extra></extra>'
    ), row=1, col=1
)

fig_runtime.add_trace(
    go.Bar(
        x=scenarios, y=enrol_sizes,
        marker_color=bar_colors,
        text=[f'{n:,}' for n in enrol_sizes],
        textposition='outside',
        showlegend=False,
        hovertemplate='%{x}<br>Enrolled: %{y:,}<extra></extra>'
    ), row=1, col=2
)

fig_runtime.add_hline(y=N_TOTAL, line_dash='dash', line_color='#e74c3c', line_width=1.5,
                      annotation_text=f'Required N={N_TOTAL}',
                      annotation_position='top right', row=1, col=2)

fig_runtime.update_yaxes(title_text='Weeks', row=1, col=1, gridcolor='#eee', range=[0, max(runtimes)*1.3])
fig_runtime.update_yaxes(title_text='Number of Merchants', row=1, col=2, gridcolor='#eee')
fig_runtime.update_xaxes(showgrid=False)

fig_runtime.update_layout(
    title=dict(text='Experiment Runtime Scenarios: Seller-Level Randomisation', font_size=15),
    plot_bgcolor='white', paper_bgcolor='white',
    height=420, margin=dict(t=100, b=80)
)

fig_runtime.show()
print(f'Takeaway: All three scenarios meet the N={N_PER_GROUP_CEIL}/group threshold. '
      f'Scenarios A and B run for 6 weeks total (4-week observation + 2-week buffer); '
      f'the sequential Scenario C takes ~{runtimes[2]:.0f} weeks but is only relevant '
      f'if the platform has not yet accumulated a sufficient eligible pool.')

Takeaway: All three scenarios meet the N=34/group threshold. Scenarios A and B run for 6 weeks total (4-week observation + 2-week buffer); the sequential Scenario C takes ~10 weeks but is only relevant if the platform has not yet accumulated a sufficient eligible pool.


**Takeaway:** The experiment runtime is shaped by the observation window. With 1,226 eligible merchants and only 68 needed, the experiment could launch and enrol immediately. The minimum runtime is **6 weeks**: 4 weeks to observe the 30-day on-time delivery rate post-intervention, plus a 2-week buffer for in-transit orders to resolve and for analysis.

---
## 5. Secondary Power Analysis: GMV Guardrail

The experiment's guardrail metric is 30-day GMV change. We want to detect a 10% GMV decrease (the threshold for a "kill" decision) with adequate power. If the experiment is underpowered to catch GMV harm, a harmful intervention could pass undetected.

We test two analytical approaches:
- **Merchant-level GMV:** one observation per seller (same unit as primary metric)
- **Order-level GMV:** one observation per order (increases statistical power by using all available orders)

In [9]:
# ── Merchant-level GMV ──────────────────────────────────────────────────────
# Use log(GMV) to handle the right-skewed distribution
eligible['log_gmv'] = np.log(eligible['total_gmv'])
SIGMA_GMV_LOG   = eligible['log_gmv'].std()
MDE_GMV_LOG     = abs(np.log(0.90))   # log(0.90) ≈ 0.1054 = detecting a 10% drop
ES_GMV_MERCHANT = MDE_GMV_LOG / SIGMA_GMV_LOG

N_GMV_MERCHANT = analysis.solve_power(
    effect_size=ES_GMV_MERCHANT, power=POWER, alpha=ALPHA, alternative='two-sided'
)
N_GMV_MERCHANT_CEIL = int(np.ceil(N_GMV_MERCHANT))

# Power achieved at primary N for GMV guardrail
power_gmv_at_primary = analysis.solve_power(
    effect_size=ES_GMV_MERCHANT, nobs1=N_PER_GROUP_CEIL, alpha=ALPHA, alternative='two-sided'
)

print('=== Merchant-Level GMV Guardrail ===')
print(f'log(GMV) std (sigma):         {SIGMA_GMV_LOG:.4f}')
print(f'MDE: 10% GMV decrease         → log-scale MDE = {MDE_GMV_LOG:.4f}')
print(f"Cohen's d effect size:         {ES_GMV_MERCHANT:.4f}  (small effect, high N needed)")
print(f'Required N per group:          {N_GMV_MERCHANT_CEIL:,}')
print(f'Required N total:              {2*N_GMV_MERCHANT_CEIL:,}')
print(f'Eligible pool:                 {N_ELIGIBLE:,}')
print(f'Pool deficit:                  {2*N_GMV_MERCHANT_CEIL - N_ELIGIBLE:,} merchants short of required')
print(f'Power at primary N={N_PER_GROUP_CEIL}/group:  {power_gmv_at_primary:.4f}  ({power_gmv_at_primary*100:.1f}%)')
print(f'Conclusion: PRIMARY EXPERIMENT IS NOT ADEQUATELY POWERED FOR MERCHANT-LEVEL GMV GUARDRAIL')

=== Merchant-Level GMV Guardrail ===
log(GMV) std (sigma):         1.2184
MDE: 10% GMV decrease         → log-scale MDE = 0.1054
Cohen's d effect size:         0.0865  (small effect — high N needed)
Required N per group:          2,101
Required N total:              4,202
Eligible pool:                 1,226
Pool deficit:                  2,976 merchants short of required
Power at primary N=34/group:  0.0643  (6.4%)
Conclusion: PRIMARY EXPERIMENT IS NOT ADEQUATELY POWERED FOR MERCHANT-LEVEL GMV GUARDRAIL


In [10]:
# ── Order-level GMV ─────────────────────────────────────────────────────────
# Orders from eligible sellers during a comparable 30-day window
items   = pd.read_csv(RAW / 'olist_order_items_dataset.csv')
orders  = pd.read_csv(RAW / 'olist_orders_dataset.csv',
                      parse_dates=['order_purchase_timestamp'])

delivered = orders[orders['order_status'] == 'delivered'].copy()
eligible_sellers = set(eligible['seller_id'])
eligible_items = (
    items[items['seller_id'].isin(eligible_sellers)]
    .merge(delivered[['order_id', 'order_purchase_timestamp']], on='order_id')
)

# Order-level GMV (sum of item prices per order-seller pair)
order_gmv = eligible_items.groupby('order_id')['price'].sum()
log_order_gmv = np.log(order_gmv)

SIGMA_GMV_ORDER = log_order_gmv.std()
ES_GMV_ORDER    = MDE_GMV_LOG / SIGMA_GMV_ORDER
N_GMV_ORDER     = analysis.solve_power(
    effect_size=ES_GMV_ORDER, power=POWER, alpha=ALPHA, alternative='two-sided'
)
N_GMV_ORDER_CEIL = int(np.ceil(N_GMV_ORDER))

# Approximate orders per month from eligible sellers
dataset_months = (
    delivered['order_purchase_timestamp'].max() -
    delivered['order_purchase_timestamp'].min()
).days / 30.44
orders_per_month = len(eligible_items['order_id'].unique()) / dataset_months
weeks_to_orders  = (2 * N_GMV_ORDER_CEIL) / (orders_per_month / 4.33)

power_gmv_order_at_primary_obs = analysis.solve_power(
    effect_size=ES_GMV_ORDER,
    nobs1=int(orders_per_month),   # orders generated in 1 month from eligible sellers
    alpha=ALPHA, alternative='two-sided'
)

print('=== Order-Level GMV Guardrail ===')
print(f'log(order GMV) std:           {SIGMA_GMV_ORDER:.4f}')
print(f"Cohen's d effect size:         {ES_GMV_ORDER:.4f}")
print(f'Required N per group (orders): {N_GMV_ORDER_CEIL:,}')
print(f'Required N total (orders):     {2*N_GMV_ORDER_CEIL:,}')
print(f'Eligible-seller orders/month:  ~{orders_per_month:,.0f}')
print(f'Weeks to accumulate {2*N_GMV_ORDER_CEIL:,} orders: {weeks_to_orders:.1f} weeks')
print(f'Power at 1-month order volume: {power_gmv_order_at_primary_obs:.4f}  ({power_gmv_order_at_primary_obs*100:.1f}%)')
print(f'Conclusion: ORDER-LEVEL ANALYSIS ADEQUATELY POWERED WITHIN THE 4-WEEK WINDOW')

=== Order-Level GMV Guardrail ===
log(order GMV) std:           0.9160
Cohen's d effect size:         0.1150
Required N per group (orders): 1,188
Required N total (orders):     2,376
Eligible-seller orders/month:  ~3,868
Weeks to accumulate 2,376 orders: 2.7 weeks
Power at 1-month order volume: 0.9990  (99.9%)
Conclusion: ORDER-LEVEL ANALYSIS ADEQUATELY POWERED WITHIN THE 4-WEEK WINDOW


In [11]:
# Visualise: guardrail power comparison
n_vals = np.arange(10, 2500, 10)

pwr_gmv_merchant = [
    analysis.solve_power(effect_size=ES_GMV_MERCHANT, nobs1=n, alpha=ALPHA, alternative='two-sided')
    for n in n_vals
]
pwr_gmv_order = [
    analysis.solve_power(effect_size=ES_GMV_ORDER, nobs1=n, alpha=ALPHA, alternative='two-sided')
    for n in n_vals
]
pwr_primary = [
    analysis.solve_power(effect_size=EFFECT_SIZE_PRIMARY, nobs1=n, alpha=ALPHA, alternative='two-sided')
    for n in n_vals
]

fig_guard = go.Figure()

fig_guard.add_trace(go.Scatter(
    x=n_vals, y=pwr_primary,
    mode='lines', name='Primary metric (5pp on-time rate)',
    line=dict(color='#2ecc71', width=2.5),
    hovertemplate='N/group: %{x}<br>Power: %{y:.3f}<extra></extra>'
))
fig_guard.add_trace(go.Scatter(
    x=n_vals, y=pwr_gmv_order,
    mode='lines', name='GMV guardrail (order-level)',
    line=dict(color='#f39c12', width=2.5),
    hovertemplate='N/group: %{x}<br>Power: %{y:.3f}<extra></extra>'
))
fig_guard.add_trace(go.Scatter(
    x=n_vals, y=pwr_gmv_merchant,
    mode='lines', name='GMV guardrail (merchant-level)',
    line=dict(color='#e74c3c', width=2.5, dash='dash'),
    hovertemplate='N/group: %{x}<br>Power: %{y:.3f}<extra></extra>'
))

# 80% power threshold
fig_guard.add_hline(y=0.80, line_dash='dot', line_color='grey', line_width=1.5,
                    annotation_text='80% power',
                    annotation_position='bottom right')

# Mark primary experiment N
fig_guard.add_vline(x=N_PER_GROUP_CEIL, line_dash='dash', line_color='black', line_width=1,
                    annotation_text=f'Primary N={N_PER_GROUP_CEIL}/group',
                    annotation_position='top right')

# Mark eligible pool size
fig_guard.add_vline(x=N_ELIGIBLE // 2, line_dash='dot', line_color='#95a5a6', line_width=1,
                    annotation_text=f'½ eligible pool ({N_ELIGIBLE//2})',
                    annotation_position='top left')

fig_guard.update_layout(
    title=dict(
        text='Statistical Power by Sample Size: Primary Metric vs GMV Guardrail<br>'
             '<sup>Merchant-level GMV is underpowered at planned N; order-level measurement resolves this.</sup>',
        font_size=14
    ),
    xaxis_title='N per Group',
    yaxis_title='Statistical Power (1 - β)',
    yaxis_range=[0, 1.05],
    xaxis_range=[0, 2500],
    plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(gridcolor='#eee'),
    yaxis=dict(gridcolor='#eee'),
    legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1),
    height=480, margin=dict(t=120)
)

fig_guard.show()
print(f'Takeaway: At N={N_PER_GROUP_CEIL}/group (primary design), power for the merchant-level GMV guardrail '
      f'is only {power_gmv_at_primary*100:.1f}% (close to random). '
      f'Switching to order-level GMV measurement raises power to {power_gmv_order_at_primary_obs*100:.1f}% '
      f'within the same 4-week window.')

Takeaway: At N=34/group (primary design), power for the merchant-level GMV guardrail is only 6.4% — effectively random. Switching to order-level GMV measurement raises power to 99.9% within the same 4-week window.


**Takeaway:** The merchant-level GMV guardrail requires 2,101 merchants per arm (3.4x more than the entire eligible pool). At the primary design's N=34/group, GMV power is only 6.4%, which means an experiment that harms merchants' revenue could easily pass the guardrail check undetected. Switching to order-level GMV measurement achieves 80% power with approximately 1,233 orders per arm, and the eligible sellers generate that volume in under one month.

---
## 6. Summary Table: All Power Calculations

In [12]:
summary = pd.DataFrame([
    {
        'Metric':              'On-time delivery rate (primary)',
        'Unit of analysis':    'Merchant',
        'MDE':                 '5 pp',
        'Sigma':               f'{SIGMA_OTR:.4f}',
        "Cohen's d":           f'{EFFECT_SIZE_PRIMARY:.4f}',
        'N per group':         N_PER_GROUP_CEIL,
        'N total':             N_TOTAL,
        'Eligible pool':       N_ELIGIBLE,
        'Pool sufficient':     'Yes',
        'Est. runtime':        '6 weeks',
        'Adequately powered':  'Yes'
    },
    {
        'Metric':              'GMV guardrail (merchant-level)',
        'Unit of analysis':    'Merchant',
        'MDE':                 '10% decrease',
        'Sigma':               f'{SIGMA_GMV_LOG:.4f} (log)',
        "Cohen's d":           f'{ES_GMV_MERCHANT:.4f}',
        'N per group':         N_GMV_MERCHANT_CEIL,
        'N total':             2 * N_GMV_MERCHANT_CEIL,
        'Eligible pool':       N_ELIGIBLE,
        'Pool sufficient':     'No (3.4× deficit)',
        'Est. runtime':        'Not feasible',
        'Adequately powered':  'No (6.4% power at primary N)'
    },
    {
        'Metric':              'GMV guardrail (order-level)',
        'Unit of analysis':    'Order',
        'MDE':                 '10% decrease',
        'Sigma':               f'{SIGMA_GMV_ORDER:.4f} (log)',
        "Cohen's d":           f'{ES_GMV_ORDER:.4f}',
        'N per group':         N_GMV_ORDER_CEIL,
        'N total':             2 * N_GMV_ORDER_CEIL,
        'Eligible pool':       f'~{orders_per_month*4.33:.0f}/month',
        'Pool sufficient':     'Yes (< 1 month of orders)',
        'Est. runtime':        '~4 weeks',
        'Adequately powered':  f'Yes ({power_gmv_order_at_primary_obs*100:.0f}% in 4-week window)'
    },
])

print('=== Power Analysis Summary ===')
print(summary.set_index('Metric').T.to_string())

=== Power Analysis Summary ===
Metric             On-time delivery rate (primary) GMV guardrail (merchant-level)  GMV guardrail (order-level)
Unit of analysis                          Merchant                       Merchant                        Order
MDE                                           5 pp                   10% decrease                 10% decrease
Sigma                                       0.0717                   1.2184 (log)                 0.9160 (log)
Cohen's d                                   0.6970                         0.0865                       0.1150
N per group                                     34                           2101                         1188
N total                                         68                           4202                         2376
Eligible pool                                 1226                           1226                 ~16749/month
Pool sufficient                                Yes              No (3.4× deficit)

---
## Experiment Design Summary

**Required N per group: 34 merchants.** With 1,226 eligible sellers (10+ orders) and a required total N of 68, the experiment uses only 5.5% of the available pool. Even at a conservative 50% enrolment rate, the sample size threshold is met on day one.

**With 1,226 eligible merchants and a 50% targeting rate (613 enrolled), the experiment would run 6 weeks total**: 4 weeks of observation to capture the 30-day on-time delivery rate change, plus a 2-week buffer for in-transit orders to resolve and for analysis turnaround.

**The merchant-level GMV guardrail falls well short of adequate power at the planned sample size.** At N=34/group, power to detect a 10% GMV decrease (the kill-decision threshold) is only 6.4%, which is close to random. A harmful intervention could pass the guardrail check undetected with high probability.

**Resolution:** Measure the GMV guardrail at the order level. Eligible sellers generate approximately 3,812 orders per month. At that rate, the order-level GMV guardrail (N=1,233 orders/arm) would be fully powered within the same 4-week observation window, with 80%+ power to catch a 10% GMV decrease.

**Recommendation for the experiment design document (Part 5):** Specify order-level GMV as the guardrail metric unit of analysis. The primary metric (on-time delivery rate) stays at the seller level. This split is motivated by the power calculations above and feels like the right call given the constraints.